# Engine v3 — Sunfish

**AI Algorithms in Chess, Fall 2026 — MATH 4900-002.** Transposition tables and iterative deepening.

Sunfish is a whole chess engine in one 662-line Python file. This notebook does the three tasks in order:

| Task | What |
|---|---|
| 1 | Read the transposition table and iterative deepening — the lines are printed below, annotated in [`notes.md`](../engines/v3-sunfish/notes.md) |
| 2 | Run the engine — two games against Stockfish, at 0.2 s and at 2 s a move, plus the clock experiment |
| 3 | Make changes and measure — a table and iterative deepening written by hand, and `tp_move` switched on → off |

The engine lives in `engines/v3-sunfish/sunfish/` (cloned with `git clone https://github.com/thomasahle/sunfish.git`);
the harness and the two experiments live in `scripts/`, so every cell here is a few lines that call them.
Open this notebook from the portfolio root or from `notebooks/` — the setup cell moves to the root either way.

Sunfish plays **White** in every game. Write-up: [`engines/v3-sunfish/RECIPE.md`](../engines/v3-sunfish/RECIPE.md).

In [1]:
import os
import sys

if not os.path.isdir("scripts") and os.path.isdir(os.path.join("..", "scripts")):
    os.chdir("..")                       # opened from notebooks/ -- work from the portfolio root instead
sys.path.insert(0, os.path.abspath("scripts"))

import chess

import play_sunfish
import run_experiments
import tt_search

print("working in     ", os.path.basename(os.getcwd()))
print("python         ", sys.version.split()[0])
print("python-chess   ", chess.__version__)
print("sunfish.py     ", os.path.relpath(play_sunfish.find_sunfish()))
print("stockfish      ", play_sunfish.find_stockfish() or "NOT FOUND -- set STOCKFISH_PATH")

working in      portfolio
python          3.14.2
python-chess    1.11.2
sunfish.py      engines\v3-sunfish\sunfish\sunfish.py
stockfish       C:/Users/hwang/Downloads/stockfish/stockfish-windows-x86-64-universal.exe


## Task 1 — read Sunfish's algorithm

`sunfish.py` is one file, so the two ideas are a few lines each. The cell below prints them
straight out of the clone, so the line numbers in my notes are checkable. All 25 are annotated
in [`notes.md`](../engines/v3-sunfish/notes.md).

In [2]:
path = play_sunfish.find_sunfish()
lines = open(path, encoding="utf-8").read().splitlines()

GROUPS = [
    ("the evaluation", [32, 33, 85, 86, 546]),
    ("the score range", [160, 169]),
    ("what an entry is", [320, 321]),
    ("the two tables", [324, 326]),
    ("the search, and the depth test in the key", [329, 380, 399]),
    ("when a stored entry may be reused", [400, 401]),
    ("the remembered move: read, tried first, written", [410, 440, 493]),
    ("the base move order", [444]),
    ("the store, and eviction", [529, 531]),
    ("iterative deepening, and the clock", [552, 558, 562]),
]

for title, numbers in GROUPS:
    print("--- %s" % title)
    for n in numbers:
        print("%4d| %s" % (n, lines[n - 1]))
    print()

--- the evaluation
  32| piece = {"P": 100, "N": 280, "B": 320, "R": 479, "Q": 929, "K": 60000}
  33| pst = {
  85|     padrow = lambda row: (0,) + tuple(x + piece[k] for x in row) + (0,)
  86|     pst[k] = sum((padrow(table[i * 8 : i * 8 + 8]) for i in range(8)), ())
 546|         pst["K"] = K_MID if "Q" in pos.board and "q" in pos.board else K_END

--- the score range
 160| MATE_UPPER = piece["K"] + 10 * piece["Q"]
 169| EVAL_ROUGHNESS = 15

--- what an entry is
 320| # lower <= s(pos) <= upper
 321| Entry = namedtuple("Entry", "lower upper")

--- the two tables
 324| class Searcher:
 326|         self.tp_score, self.tp_move, self.history = {}, {}, set()

--- the search, and the depth test in the key
 329|     def bound(self, pos, gamma, depth, root=False):
 380|         depth = max(depth, 0)
 399|             entry = self.tp_score.get((pos, depth), Entry(-MATE_UPPER, MATE_UPPER))

--- when a stored entry may be reused
 400|             if entry.lower >= gamma: return entry.lower
 40

**In my own words.** Sunfish is a minimalist engine whose evaluation is nothing but piece-square
tables and whose search is given a clock rather than a depth; its two ideas are a transposition
table and iterative deepening.

`tp_score` stores, for a position **and the depth it was searched to**, a pair of numbers
`Entry(lower, upper)` that the true score lies between; `tp_move` stores, for the position alone,
the best move found there.

- **The depth test.** An entry may only be reused by a search that wants no more depth than the
  entry was given. Sunfish does this test *by putting the depth in the key* —
  `self.tp_score[pos, depth]` — so a depth-3 entry is simply never read by a depth-4 probe, where
  most engines keep one entry per position and compare `stored_depth >= depth` instead.
- **Exact value vs bound.** An exact value is the score and can be returned as the answer. A bound
  only rules out one side: a lower bound ends the search when it already reaches the window
  (`entry.lower >= gamma`), an upper bound ends it when it already falls short
  (`entry.upper < gamma`), and if neither fires the position has to be searched again — the entry
  is then only a hint about which move to try.
- `tp_move` needs **no** depth test, because a remembered move changes only the order moves are
  tried in, never the value of the sub-tree. That is why it is sound to take a move from a
  depth-1 pass into a depth-4 one, which is the whole trick behind iterative deepening.

## Task 2 — run the engine

First the handshake: Sunfish started as a UCI engine and answering. It has to be launched from
inside its own folder, because `main()` imports `sunfish_ui.uci`.

In [3]:
engine = play_sunfish.open_sunfish()
print("engine.id =", engine.id)
engine.quit()

engine.id = {'name': 'sunfish 2026'}


### A full game, 0.2 s a move

Black is Stockfish, rating-limited to 1350 with 0.1 s a move — the same opponent settings as v1
and v2, so the three engines are comparable. Every move is printed with the depth, nodes and score
Sunfish reports over UCI.

In [4]:
board_fast = play_sunfish.play_sunfish(
    opponent="stockfish", seconds=0.2, elo=1350,
    out="engines/v3-sunfish/games/v3-sunfish-vs-stockfish-0.2s.pgn")

engine.id = {'name': 'sunfish 2026'}


White: Sunfish (0.2 s a move)    Black: Stockfish (1350)

    1. SUN f4       ( 0.2s)  depth  2    1,885 nodes  score +17


       opp c6       ( 0.1s)
    2. SUN e3       ( 0.2s)  depth  3    2,515 nodes  score +40


       opp a6       ( 0.1s)


    3. SUN d4       ( 0.3s)  depth  4    3,996 nodes  score +58
       opp e6       ( 0.1s)


    4. SUN Nf3      ( 0.2s)  depth  4    2,520 nodes  score +50
       opp c5       ( 0.1s)


... (24 outputs left out)

   29. SUN Qxc7     ( 0.2s)  depth  6    3,021 nodes  score +1818
       opp f5       ( 0.1s)


   30. SUN exf6     ( 0.2s)  depth  5    2,479 nodes  score +2039
       opp gxf6     ( 0.0s)
   31. SUN Nxb4     ( 0.1s)  depth  4    2,212 nodes  score +1899


       opp f5       ( 0.1s)


   32. SUN Qg7#     ( 0.4s)  depth  7      531 nodes  score +48013

. . . . . r k .
. . . . . . Q p
. . . . . . . .
. . . . . p . .
. N . . R . . .
. P . . . . P .
. B P . . . . P
. . . . . . K .

63 plies in 10s.  Result 1-0  (checkmate)   Sunfish searched 70,396 nodes in all.
saved game -> engines\v3-sunfish\games\v3-sunfish-vs-stockfish-0.2s.pgn


### The same game, 2 s a move

Ten times the clock. Watch the depth column rather than the result.

In [5]:
board_slow = play_sunfish.play_sunfish(
    opponent="stockfish", seconds=2.0, elo=1350,
    out="engines/v3-sunfish/games/v3-sunfish-vs-stockfish-2s.pgn")

engine.id = {'name': 'sunfish 2026'}


White: Sunfish (2 s a move)    Black: Stockfish (1350)



    1. SUN d4       ( 1.5s)  depth  4   18,312 nodes  score +14
       opp Nf6      ( 0.1s)


    2. SUN Nc3      ( 1.5s)  depth  5   18,263 nodes  score +9
       opp d5       ( 0.1s)


    3. SUN e3       ( 1.9s)  depth  5   25,990 nodes  score +23
       opp Bf5      ( 0.1s)


    4. SUN Nf3      ( 2.1s)  depth  6   12,574 nodes  score +33
       opp Bg6      ( 0.1s)


... (25 outputs left out)

   30. SUN d5       ( 1.5s)  depth  7   32,427 nodes  score +2014
       opp Rf8      ( 0.1s)


   31. SUN Rxg7+    ( 1.4s)  depth  8   18,960 nodes  score +2778
       opp Kh8      ( 0.1s)


   32. SUN Be6      ( 2.1s)  depth  9   18,365 nodes  score +2606
       opp Ra8      ( 0.1s)


   33. SUN Rd7#     ( 2.0s)  depth  9    8,056 nodes  score +48043

r . . . . . . k
. . . R . . . .
. . . . B . p .
. . . P . . P .
. . . . . P . p
. P B P P . . .
. P . . . . . P
. . . . . R K .

65 plies in 60s.  Result 1-0  (checkmate)   Sunfish searched 641,511 nodes in all.
saved game -> engines\v3-sunfish\games\v3-sunfish-vs-stockfish-2s.pgn


### Task 2.3 — the clock experiment

One position, four clocks. Iterative deepening means the answer is ready at every instant, so the
question is what the extra time adds to it: depth, nodes, and sometimes the move.

In [6]:
clock_rows = play_sunfish.clock_experiment()

 0.2 s: Qb3    depth  3    2,228 nodes  score -38   (0.2s)


 1.0 s: Qb3    depth  5   12,849 nodes  score -37   (0.9s)


 2.0 s: Qb3    depth  5   18,179 nodes  score -24   (1.3s)


 3.0 s: Qb3    depth  7   29,971 nodes  score +69   (2.1s)


### Against me

Played at the keyboard, in SAN:

```bash
python scripts/play_sunfish.py --opponent you
```

Paste the moves I played as Black into `MY_MOVES` below and the cell replays the game, so a
notebook re-run reproduces it instead of stopping for input. With `MY_MOVES` empty the cell says
so and plays nothing.

In [7]:
MY_MOVES = ""     # Black's moves, in SAN: "c5 Nc6 d5 ..."

if MY_MOVES.strip():
    board_me = play_sunfish.play_sunfish(
        opponent="you", seconds=0.2, moves=MY_MOVES,
        out="engines/v3-sunfish/games/v3-sunfish-vs-me.pgn")
else:
    print("No moves recorded yet.")
    print("Play it at the keyboard with:  python scripts/play_sunfish.py --opponent you")
    print("then paste Black's moves into MY_MOVES above and re-run this cell.")

No moves recorded yet.
Play it at the keyboard with:  python scripts/play_sunfish.py --opponent you
then paste Black's moves into MY_MOVES above and re-run this cell.


## Task 3 — make changes and measure

### 3.1 — the table and iterative deepening, written by hand

On top of last week's alpha-beta, in `scripts/tt_search.py`: a transposition table keyed by
python-chess's Zobrist hash, storing the depth, the value, whether the value is exact or a bound,
and the best move; and iterative deepening, depths 1 to 4 sharing one table.

Four searches of last week's three positions at depth 4 — alpha-beta alone, plus the table, plus
deepening, plus the remembered best move tried first. **The move and the value must be identical on
every row**; the node counts must not be. The cell asserts the first part rather than leaving it to
the eye. Nothing here starts an engine.

In [8]:
rows_all = tt_search.node_table(depth=4)

start        alpha-beta alone     17,862 nodes       0 hits   Nf3        -5.0


start        + table              15,807 nodes     398 hits   Nf3        -5.0


start        + deepening          18,077 nodes     408 hits   Nf3        -5.0


start        + best move first     8,013 nodes     138 hits   Nf3        -5.0



Italian      alpha-beta alone     19,722 nodes       0 hits   Bg5       -93.0


Italian      + table              19,013 nodes     362 hits   Bg5       -93.0


... (2 outputs left out)

open middle  alpha-beta alone     12,219 nodes       0 hits   Nfd2     -197.0


open middle  + table              11,745 nodes     440 hits   Nfd2     -197.0


open middle  + deepening          13,089 nodes     440 hits   Nfd2     -197.0


open middle  + best move first    11,310 nodes     418 hits   Nfd2     -197.0



### 3.2 — my one change: `tp_move` on → off

The change is Sunfish's own line 440: remember the best move in the table and try it first on the
next pass. In my search that is the single flag `use_tt_move`; everything else is identical — same
evaluation, same positions, same depth, same base move order, same table.

In [9]:
rows_change = run_experiments.node_table_before_after(depth=4)

start        TT + deepening (before)      18,077 nodes     408 hits   Nf3        -5.0


start        + tp_move first (after)       8,013 nodes     138 hits   Nf3        -5.0
             after / before                 44.3% of the nodes left, so 55.7% saved



Italian      TT + deepening (before)      21,141 nodes     373 hits   Bg5       -93.0


Italian      + tp_move first (after)       9,082 nodes     365 hits   Bg5       -93.0
             after / before                 43.0% of the nodes left, so 57.0% saved



open middle  TT + deepening (before)      13,089 nodes     440 hits   Nfd2     -197.0


open middle  + tp_move first (after)      11,310 nodes     418 hits   Nfd2     -197.0
             after / before                 86.4% of the nodes left, so 13.6% saved



### 3.3 — the port: Greedy with Sunfish's piece-square tables

Thirty games, colours alternating. Both bots look one ply ahead and take the move that leaves the
best material; the only difference is that the ported bot breaks a tie in material with Sunfish's
piece-square table instead of at random.

In [10]:
points, decisive, unfinished = run_experiments.match(games=30, max_plies=120, seed=1)
print("\n  port scored %.1f / 30  (%d decisive, %d unfinished at 120 plies and scored a draw)"
      % (points, decisive, unfinished))

  game  1  port as White  1-0      port scores 1.0   (running 1.0/1)
  game  2  port as Black  1-0      port scores 0.0   (running 1.0/2)


  game  3  port as White  1-0      port scores 1.0   (running 2.0/3)


  game  4  port as Black  1/2-1/2  port scores 0.5   (running 2.5/4)


  game  5  port as White  0-1      port scores 0.0   (running 2.5/5)
  game  6  port as Black  1-0      port scores 0.0   (running 2.5/6)


  game  7  port as White  1/2-1/2  port scores 0.5   (running 3.0/7)
  game  8  port as Black  1/2-1/2  port scores 0.5   (running 3.5/8)


  game  9  port as White  1/2-1/2  port scores 0.5   (running 4.0/9)


... (4 outputs left out)

  game 19  port as White  0-1      port scores 0.0   (running 9.0/19)
  game 20  port as Black  1/2-1/2  port scores 0.5   (running 9.5/20)
  game 21  port as White  1-0      port scores 1.0   (running 10.5/21)


  game 22  port as Black  1/2-1/2  port scores 0.5   (running 11.0/22)
  game 23  port as White  1-0      port scores 1.0   (running 12.0/23)
  game 24  port as Black  1/2-1/2  port scores 0.5   (running 12.5/24)


  game 25  port as White  1/2-1/2  port scores 0.5   (running 13.0/25)
  game 26  port as Black  1/2-1/2  port scores 0.5   (running 13.5/26)
  game 27  port as White  0-1      port scores 0.0   (running 13.5/27)
  game 28  port as Black  0-1      port scores 1.0   (running 14.5/28)


  game 29  port as White  1/2-1/2  port scores 0.5   (running 15.0/29)
  game 30  port as Black  1/2-1/2  port scores 0.5   (running 15.5/30)

  port scored 15.5 / 30  (11 decisive, 1 unfinished at 120 plies and scored a draw)


In [11]:
saved = run_experiments.write_results(rows_change, 30, points, decisive, unfinished)
print()
print(open("engines/v3-sunfish/results.csv", encoding="utf-8").read())

wrote engines\v3-sunfish\results.csv

experiment,label,depth,nodes,table_hits,move,score,games,points_for_change,decisive
tt_depth4,start -- TT + deepening (before),4,18077,408,Nf3,-5.0,,,
tt_depth4,start -- + tp_move first (after),4,8013,138,Nf3,-5.0,,,
tt_depth4,Italian -- TT + deepening (before),4,21141,373,Bg5,-93.0,,,
tt_depth4,Italian -- + tp_move first (after),4,9082,365,Bg5,-93.0,,,
tt_depth4,open middle -- TT + deepening (before),4,13089,440,Nfd2,-197.0,,,
tt_depth4,open middle -- + tp_move first (after),4,11310,418,Nfd2,-197.0,,,
port_match,"pst-greedy vs plain greedy, colours alternating (1 unfinished at 120 plies)",,,,,,30,15.5,11



## What I found

- **A transposition table on its own is nearly free and nearly useless** — it saved 4–12% of the
  nodes at depth 4. **Iterative deepening on its own costs more than it saves**: re-searching
  depths 1 to 3 is 11–14% more nodes than going straight to 4.
- **Together, through the move order, they pay.** Trying the remembered best move first cut the
  depth-4 search by 56% at the start and 57% in the Italian, and by 14% in the open middlegame,
  where captures already dominate the base ordering. That is why Sunfish keeps `tp_move` in a
  separate table from `tp_score`, keyed by the position with no depth attached.
- **And it changes nothing.** The move and the value are the same on all twelve rows. A table and a
  move order are bookkeeping, not a second opinion.
- **The clock buys depth and an opinion, not a different move.** `Qb3` at every clock from the
  Italian middlegame, but the score moved from −38 at depth 3 to +69 at depth 7. In the games,
  ten times the clock bought about two plies: median depth 4 at 0.2 s, 6 at 2 s.
- **The port is level.** 15.5 / 30 against plain Greedy — thirty games cannot support any claim
  stronger than "if the tables help a one-ply bot, the effect is smaller than thirty games can see".
- **The two games here are not repeatable.** Sunfish is given a clock, not a depth, and Stockfish at
  `UCI_LimitStrength` randomises; Sunfish won both of these and lost both when I ran the same two
  games from the command line an hour earlier. One game each way says nothing about either engine.
- **Its weakness is the evaluation, not the search.** A piece-square table knows nothing about king
  safety, and Sunfish castled and then pushed `9. g4`, `10. g5`, `11. f4` in front of its own king
  in the 2 s game — the tables pay for those squares and nothing charges for the holes. Against a
  1350 opponent it got away with it; in the command-line games it was mated for exactly that.

Full write-up, with the commands and what broke:
[`engines/v3-sunfish/RECIPE.md`](../engines/v3-sunfish/RECIPE.md).